# Generating Captions

Generating captions from trained models for qualitataive comparison.

In [ ]:
from pathlib import Path
from types import SimpleNamespace

import torch
from PIL import Image
from torchvision import transforms

from notebooks.ic.ic_utils import generate_batch_token_ids

# imports from your project
# adapt only if these are in another module
# from dataloader import build_tokenizer_from_split
# from model_file import ShowAttendTellCaptioner


# ==================================================
# 1. Paths and settings
# ==================================================
checkpoint_path = Path("checkpoints/model2_20260525-190721_20260525_195713_last.pth")

data_dir = Path("data")

image_paths = [
    Path("data/images/1015118661_980735411b.jpg"),
    Path("data/images/1022454332_6af2c1449a.jpg"),
    Path("data/images/102455176_5f8ead62d5.jpg"),
    Path("data/images/1030985833_b0902ea560.jpg"),
    Path("data/images/104136873_5b5d41be75.jpg"),
    Path("data/images/1042020065_fb3d3ba5ba.jpg"),
    Path("data/images/1055623002_8195a43714.jpg"),
    Path("data/images/106490881_5a2dd9b7bd.jpg"),
]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# ==================================================
# 2. Validation / test transform
# ==================================================
transform = transforms.Compose([
    transforms.Lambda(_convert_to_rgb),
    transforms.Lambda(_resize_with_padding),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])


# ==================================================
# 3. Load checkpoint
# ==================================================
checkpoint = torch.load(checkpoint_path, map_location=device)

cfg_dict = checkpoint["model_config"]

full_cfg = SimpleNamespace()
full_cfg.model = model_cfg

full_cfg.train = SimpleNamespace(
    device=str(device),
    non_blocking=device.type == "cuda",
)

full_cfg.generation = SimpleNamespace(
    max_len=MAX_LEN,
    bos_idx=model_cfg.bos_idx,
    eos_idx=model_cfg.eos_idx,
    pad_idx=model_cfg.pad_idx,
)

full_cfg.non_blocking = device.type == "cuda"
full_cfg.max_len = MAX_LEN
full_cfg.pad_idx = model_cfg.pad_idx
full_cfg.bos_idx = model_cfg.bos_idx
full_cfg.eos_idx = model_cfg.eos_idx

# ==================================================
# 4. Rebuild tokenizer
# ==================================================
tokenizer = build_tokenizer_from_split(
    split="train",
    data_dir=data_dir,
    min_freq=MIN_WORD_FREQ,
)

print("Tokenizer vocab size:", len(tokenizer))
print("Checkpoint vocab size:", model_cfg.vocab_size)

assert len(tokenizer) == model_cfg.vocab_size, (
    f"Tokenizer vocab size {len(tokenizer)} does not match checkpoint vocab size "
    f"{model_cfg.vocab_size}. Use the same data_dir and MIN_WORD_FREQ as during training."
)


# ==================================================
# 5. Rebuild model and load weights
# ==================================================
model = ShowAttendTellCaptioner(model_cfg)

model.load_state_dict(checkpoint["state_dict"])
model.to(device)
model.eval()

# print(hasattr(model_cfg, "non_blocking"))
# print(model_cfg.non_blocking)


# ==================================================
# 6. Caption generation
# ==================================================
@torch.no_grad()
def caption_image(model, image_path, tokenizer, cfg, device):

    image = Image.open(image_path).convert("RGB")
    image = transform(image).unsqueeze(0).to(device)

    generated_ids = generate_batch_token_ids(
        model=model,
        images=image,
        cfg=cfg,
    )[0]

    if isinstance(generated_ids, torch.Tensor):
        generated_ids = generated_ids.detach().cpu().tolist()

    caption = tokenizer.decode(generated_ids)

    caption = (
        caption.replace("<bos>", "")
        .replace("<eos>", "")
        .replace("<pad>", "")
        .strip()
    )

    return caption


# ==================================================
# 7. Run on selected images
# ==================================================
for image_path in image_paths:
    caption = caption_image(
        model=model,
        image_path=image_path,
        tokenizer=tokenizer,
        cfg=full_cfg,
        device=device,
    )

    print(f"{image_path.name}: {caption}")